# Task 3 – ML Model Portfolio
**Four Models:** Logistic Regression, Random Forest, GBT, Linear SVM
**Tuning:** CrossValidator + ParamGrid


In [1]:
# ============================================================
# TASK 3: ML Model Portfolio
# Models: Logistic Regression, Random Forest, GBT, Linear SVM
# ============================================================

from pyspark.sql import SparkSession
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,
    LinearSVC
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import time

spark = SparkSession.builder \
    .appName('7006SCN_Task3_MLPortfolio') \
    .config('spark.executor.memory', '4g') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print('SparkSession initialised')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 14:01:51 WARN Utils: Your hostname, Naveenrajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.34 instead (on interface en0)
26/06/29 14:01:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 14:01:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 14:01:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/29 14:01:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/29 14:01:51 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


SparkSession initialised


In [2]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

train_df = spark.read.parquet('./data/train_processed.parquet')
test_df = spark.read.parquet('./data/test_processed.parquet')

# Cache for faster repeated access
train_df.cache()
test_df.cache()

print(f'Training rows: {train_df.count():,}')
print(f'Test rows: {test_df.count():,}')

# Class balance check
from pyspark.sql.functions import col
train_df.groupBy('high_tip').count().show()

Training rows: 2,297,635
Test rows: 573,716
+--------+-------+
|high_tip|  count|
+--------+-------+
|       1|1407319|
|       0| 890316|
+--------+-------+



In [3]:
# ============================================================
# HELPER FUNCTION: EVALUATE MODEL
# ============================================================

def evaluate_model(predictions, model_name):
    binary_eval = BinaryClassificationEvaluator(
        labelCol='high_tip', rawPredictionCol='rawPrediction'
    )
    multi_eval = MulticlassClassificationEvaluator(
        labelCol='high_tip', predictionCol='prediction'
    )

    auc = binary_eval.evaluate(predictions)
    acc = multi_eval.evaluate(predictions, {multi_eval.metricName: 'accuracy'})
    prec = multi_eval.evaluate(predictions, {multi_eval.metricName: 'weightedPrecision'})
    rec = multi_eval.evaluate(predictions, {multi_eval.metricName: 'weightedRecall'})
    f1 = multi_eval.evaluate(predictions, {multi_eval.metricName: 'f1'})

    print(f'\n=== {model_name} Results ===')
    print(f'  AUC-ROC:   {auc:.4f}')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    return {'model': model_name, 'auc': auc, 'accuracy': acc,
            'precision': prec, 'recall': rec, 'f1': f1}

results = []

In [4]:
# ============================================================
# MODEL 1: LOGISTIC REGRESSION
# Justification: Baseline linear model; fast, interpretable,
# good for binary classification with scaled features
# ============================================================

print('Training Model 1: Logistic Regression...')
start = time.time()

lr = LogisticRegression(
    labelCol='high_tip',
    featuresCol='features',
    maxIter=100
)

lr_param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol='high_tip'),
    numFolds=3,
    seed=42
)

lr_model = lr_cv.fit(train_df)
lr_time = time.time() - start
print(f'Training time: {lr_time:.2f} seconds')

lr_preds = lr_model.transform(test_df)
lr_results = evaluate_model(lr_preds, 'Logistic Regression')
lr_results['training_time'] = lr_time
results.append(lr_results)

# Best hyperparameters
best_lr = lr_model.bestModel
print(f'Best regParam: {best_lr._java_obj.getRegParam()}')
print(f'Best elasticNetParam: {best_lr._java_obj.getElasticNetParam()}')

Training Model 1: Logistic Regression...


26/06/29 14:01:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/29 14:01:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

Training time: 21.67 seconds

=== Logistic Regression Results ===
  AUC-ROC:   0.7840
  Accuracy:  0.7957
  Precision: 0.8457
  Recall:    0.7957
  F1-Score:  0.7741
Best regParam: 0.01
Best elasticNetParam: 0.5


In [5]:
# ============================================================
# MODEL 2: RANDOM FOREST CLASSIFIER
# Justification: Ensemble method, handles non-linearity,
# resistant to overfitting, provides feature importance
# ============================================================

print('Training Model 2: Random Forest...')
start = time.time()

rf = RandomForestClassifier(
    labelCol='high_tip',
    featuresCol='features',
    seed=42
)

rf_param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol='high_tip'),
    numFolds=3,
    seed=42
)

rf_model = rf_cv.fit(train_df)
rf_time = time.time() - start
print(f'Training time: {rf_time:.2f} seconds')

rf_preds = rf_model.transform(test_df)
rf_results = evaluate_model(rf_preds, 'Random Forest')
rf_results['training_time'] = rf_time
results.append(rf_results)

best_rf = rf_model.bestModel
print(f'Best numTrees: {best_rf.getNumTrees}')
print(f'Best maxDepth: {best_rf.getMaxDepth()}')
print('Feature importances:', best_rf.featureImportances)

Training Model 2: Random Forest...


26/06/29 14:02:29 WARN DAGScheduler: Broadcasting large task binary with size 1486.6 KiB
26/06/29 14:02:31 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/06/29 14:02:32 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/06/29 14:02:34 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/06/29 14:02:53 WARN DAGScheduler: Broadcasting large task binary with size 1620.1 KiB
26/06/29 14:02:55 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/06/29 14:02:58 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/06/29 14:03:01 WARN DAGScheduler: Broadcasting large task binary with size 1173.3 KiB
26/06/29 14:03:02 WARN DAGScheduler: Broadcasting large task binary with size 8.3 MiB
26/06/29 14:03:06 WARN DAGScheduler: Broadcasting large task binary with size 1977.0 KiB
26/06/29 14:03:06 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/06/29 14:03:17 WARN DAGScheduler: Broad

Training time: 192.36 seconds


26/06/29 14:05:31 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/06/29 14:05:33 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/06/29 14:05:33 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/06/29 14:05:34 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/06/29 14:05:35 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB



=== Random Forest Results ===
  AUC-ROC:   0.7909
  Accuracy:  0.7978
  Precision: 0.8475
  Recall:    0.7978
  F1-Score:  0.7767
Best numTrees: 100
Best maxDepth: 10
Feature importances: (13,[0,1,2,3,4,5,6,7,8,9,10,11,12],[0.009201896686771502,0.014525531507138385,0.0034370705228557127,0.00017124493716970236,6.247732030885784e-07,0.00016007903650362037,0.00015974694836289662,0.008061810245134523,0.0036638976110577193,0.005015783370160422,0.0003201867424459571,0.9535574729962526,0.0017246546229439741])


In [6]:
# ============================================================
# MODEL 3: GRADIENT BOOSTED TREES (GBT)
# Justification: Sequential boosting, captures complex feature
# interactions, typically highest accuracy for tabular data
# ============================================================

print('Training Model 3: GBT Classifier...')
start = time.time()

gbt = GBTClassifier(
    labelCol='high_tip',
    featuresCol='features',
    seed=42
)

gbt_param_grid = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [20, 50]) \
    .addGrid(gbt.maxDepth, [3, 5]) \
    .addGrid(gbt.stepSize, [0.05, 0.1]) \
    .build()

gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol='high_tip'),
    numFolds=3,
    seed=42
)

gbt_model = gbt_cv.fit(train_df)
gbt_time = time.time() - start
print(f'Training time: {gbt_time:.2f} seconds')

gbt_preds = gbt_model.transform(test_df)
gbt_results = evaluate_model(gbt_preds, 'GBT Classifier')
gbt_results['training_time'] = gbt_time
results.append(gbt_results)

best_gbt = gbt_model.bestModel
print(f'Best maxIter: {best_gbt.getMaxIter()}')
print(f'Best maxDepth: {best_gbt.getMaxDepth()}')

Training Model 3: GBT Classifier...


Training time: 319.49 seconds

=== GBT Classifier Results ===
  AUC-ROC:   0.7920
  Accuracy:  0.7977
  Precision: 0.8465
  Recall:    0.7977
  F1-Score:  0.7768
Best maxIter: 50
Best maxDepth: 5


In [7]:
# ============================================================
# MODEL 4: LINEAR SVM
# Justification: Effective in high-dimensional space, uses
# hinge loss, good for binary classification problems
# ============================================================

print('Training Model 4: Linear SVM...')
start = time.time()

svm = LinearSVC(
    labelCol='high_tip',
    featuresCol='features'
)

svm_param_grid = ParamGridBuilder() \
    .addGrid(svm.regParam, [0.01, 0.1]) \
    .addGrid(svm.maxIter, [50, 100]) \
    .build()

svm_cv = CrossValidator(
    estimator=svm,
    estimatorParamMaps=svm_param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol='high_tip'),
    numFolds=3,
    seed=42
)

svm_model = svm_cv.fit(train_df)
svm_time = time.time() - start
print(f'Training time: {svm_time:.2f} seconds')

svm_preds = svm_model.transform(test_df)
svm_results = evaluate_model(svm_preds, 'Linear SVM')
svm_results['training_time'] = svm_time
results.append(svm_results)

Training Model 4: Linear SVM...
Training time: 71.25 seconds

=== Linear SVM Results ===
  AUC-ROC:   0.7530
  Accuracy:  0.7957
  Precision: 0.8468
  Recall:    0.7957
  F1-Score:  0.7739


In [9]:
# ============================================================
# MODEL SUMMARY TABLE
# ============================================================

import pandas as pd

summary_df = pd.DataFrame(results)
summary_df = summary_df.set_index('model')
summary_df['training_time'] = summary_df['training_time'].round(2)
summary_df[['accuracy', 'precision', 'recall', 'f1', 'auc']] = \
    summary_df[['accuracy', 'precision', 'recall', 'f1', 'auc']].round(4)

print('\n=== COMPLETE MODEL SUMMARY TABLE ===')
print(summary_df.to_string())

# Save predictions for Task 5
lr_preds.write.mode('overwrite').parquet('./data/lr_predictions.parquet')
rf_preds.write.mode('overwrite').parquet('./data/rf_predictions.parquet')
gbt_preds.write.mode('overwrite').parquet('./data/gbt_predictions.parquet')
svm_preds.write.mode('overwrite').parquet('./data/svm_predictions.parquet')
print('\nPredictions saved for Task 5')

# spark.stop()


=== COMPLETE MODEL SUMMARY TABLE ===
                        auc  accuracy  precision  recall      f1  training_time
model                                                                          
Logistic Regression  0.7840    0.7957     0.8457  0.7957  0.7741          21.67
Random Forest        0.7909    0.7978     0.8475  0.7978  0.7767         192.36
GBT Classifier       0.7920    0.7977     0.8465  0.7977  0.7768         319.49
Linear SVM           0.7530    0.7957     0.8468  0.7957  0.7739          71.25


26/06/29 14:16:16 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
                                                                                


Predictions saved for Task 5
